In [2]:
import pandas as pd


In [3]:
pedestrian_df= pd.read_csv("pedestrian-counting-system-monthly-counts-per-hour.csv")
                           

In [4]:
print(pedestrian_df.shape)

(1615095, 9)


In [5]:
parking_df =  pd.read_csv("on-street-parking-bay-sensors.csv")

In [6]:
print(parking_df.shape)

(6324, 6)


In [7]:
print(pedestrian_df.head())
print(pedestrian_df.info())
print(parking_df.head())
print(parking_df.info())

             ID  Location_ID Sensing_Date  HourDay  Direction_1  Direction_2  \
0   66320260809           66   2026-08-09        3           43           24   
1  132320260809          132   2026-08-09        3            3           10   
2   61020260809           61   2026-08-09        0           69           45   
3  164220260809          164   2026-08-09        2            5           12   
4   18020260809           18   2026-08-09        0            6            2   

   Total_of_Directions Sensor_Name                    Location  
0                   67       QVN_T  -37.81057846, 144.96444294  
1                   13   King335_T  -37.81267639, 144.95386444  
2                  114    RMIT14_T  -37.80767455, 144.96309114  
3                   17    Lat526_T    -37.813005, 144.95160411  
4                    8     Col12_T  -37.81344862, 144.97305353  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1615095 entries, 0 to 1615094
Data columns (total 9 columns):
 #   Column      

In [8]:
# --- Pedestrian cleaning ---
pedestrian_df = pedestrian_df.dropna(subset=['Sensor_Name', 'Location'])

pedestrian_df['Sensing_Date'] = pd.to_datetime(pedestrian_df['Sensing_Date'])
pedestrian_df['DayOfWeek'] = pedestrian_df['Sensing_Date'].dt.dayofweek  # 0=Mon
pedestrian_df['Month'] = pedestrian_df['Sensing_Date'].dt.month

pedestrian_df[['Latitude', 'Longitude']] = pedestrian_df['Location'].str.split(',', expand=True).astype(float)

print(pedestrian_df.shape)
print(pedestrian_df.isnull().sum())

(1589785, 13)
ID                     0
Location_ID            0
Sensing_Date           0
HourDay                0
Direction_1            0
Direction_2            0
Total_of_Directions    0
Sensor_Name            0
Location               0
DayOfWeek              0
Month                  0
Latitude               0
Longitude              0
dtype: int64


In [9]:
# --- Parking cleaning ---
parking_df['Lastupdated'] = pd.to_datetime(parking_df['Lastupdated'])
parking_df['Status_Timestamp'] = pd.to_datetime(parking_df['Status_Timestamp'])

parking_df[['Latitude', 'Longitude']] = parking_df['Location'].str.split(',', expand=True).astype(float)

# Target: convert Present/Unoccupied to binary for classification
parking_df['Occupied'] = parking_df['Status_Description'].map({'Present': 1, 'Unoccupied': 0})

print(parking_df.shape)
print(parking_df['Status_Description'].value_counts())
print(parking_df.isnull().sum())

(6324, 9)
Status_Description
Present       3251
Unoccupied    3073
Name: count, dtype: int64
Lastupdated             0
Status_Timestamp        0
Zone_Number           529
Status_Description      0
KerbsideID              0
Location                0
Latitude                0
Longitude               0
Occupied                0
dtype: int64


C:\Users\SS\AppData\Local\Temp\ipykernel_25300\1192171528.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parking_df['Lastupdated'] = pd.to_datetime(parking_df['Lastupdated'])
C:\Users\SS\AppData\Local\Temp\ipykernel_25300\1192171528.py:3: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parking_df['Status_Timestamp'] = pd.to_datetime(parking_df['Status_Timestamp'])


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

parking_clean = parking_df.dropna(subset=['Zone_Number'])  # drop rows missing this feature

features = ['Latitude', 'Longitude', 'Zone_Number']
X = parking_clean[features]
y = parking_clean['Occupied']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)  # max_depth limits overfitting
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# which factor mattered most
import pandas as pd
print(pd.Series(dt_model.feature_importances_, index=features).sort_values(ascending=False))

Accuracy: 0.6229508196721312
Precision: 0.6364985163204748
Recall: 0.6908212560386473
F1: 0.6625482625482626
Confusion Matrix:
 [[293 245]
 [192 429]]
Longitude      0.403108
Latitude       0.347073
Zone_Number    0.249819
dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# sample if this is slow ,1.5M rows is a lot for a first run
pedestrian_sample = pedestrian_df.sample(n=200000, random_state=42)

features_p = ['HourDay', 'DayOfWeek', 'Month', 'Latitude', 'Longitude']
X_p = pedestrian_sample[features_p]
y_p = pedestrian_sample['Total_of_Directions']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_p, y_train_p)

y_pred_p = rf_model.predict(X_test_p)

print("MAE:", mean_absolute_error(y_test_p, y_pred_p))
print("RMSE:", mean_squared_error(y_test_p, y_pred_p, squared=False))
print("R²:", r2_score(y_test_p, y_pred_p))

print(pd.Series(rf_model.feature_importances_, index=features_p).sort_values(ascending=False))

MAE: 83.09857748351162


TypeError: got an unexpected keyword argument 'squared'

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# sample if this is slow ,1.5M rows is a lot for a first run
pedestrian_sample = pedestrian_df.sample(n=200000, random_state=42)

features_p = ['HourDay', 'DayOfWeek', 'Month', 'Latitude', 'Longitude']
X_p = pedestrian_sample[features_p]
y_p = pedestrian_sample['Total_of_Directions']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_p, y_train_p)

y_pred_p = rf_model.predict(X_test_p)

print("MAE:", mean_absolute_error(y_test_p, y_pred_p))
import numpy as np
print("RMSE:", np.sqrt(mean_squared_error(y_test_p, y_pred_p)))
print("R²:", r2_score(y_test_p, y_pred_p))

print(pd.Series(rf_model.feature_importances_, index=features_p).sort_values(ascending=False))

MAE: 83.09857748351162
RMSE: 206.03625782280275
R²: 0.877521657667753
HourDay      0.325623
Longitude    0.323211
Latitude     0.236457
DayOfWeek    0.060011
Month        0.054698
dtype: float64
